## GenAI Disclosure Statement

In this course, generative AI tools were used as learning aids to support understanding
of Responsible ML concepts. All analysis, interpretations, and conclusions are the
original work of the project team.

---

# Notebook 01 — Data Preparation and Label Construction
### DNSC 6330 Responsible Machine Learning Capstone | GWU
**Audit framework:** Measurement before opinion · Diagnostics before remediation · Documentation before deployment

**Purpose:** Load 2024 HMDA LAR, apply label rule, remove leakage features, engineer features,
build protected-attribute vectors, and export train/val/test/geo_holdout splits.

**Inputs:** `../2024_lar.zip` (raw HMDA LAR)  
**Outputs:** `data/processed/*.parquet`  
**Dependencies:** None (first notebook in pipeline)


In [1]:
import sys, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import warnings
warnings.filterwarnings("ignore")
import zipfile
import hashlib
from sklearn.model_selection import train_test_split

# Define paths relative to the current working directory
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
print(f"Project root directory: {project_root}")

# Ensure local package imports resolve from the project root
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.labels import apply_label, label_distribution_table
from src.leakage import remove_leakage, get_proxy_risk_table

# Fixed seeds for reproducibility
SEED = 42
np.random.seed(SEED)

TABLES_DIR = project_root / "tables"
FIG_DIR = project_root / "figures"
DATA_ZIP = project_root / "data" / "raw" / "2024_lar.zip"
PROC_DIR = project_root / "data" / "processed"

print(f"DATA_ZIP: {DATA_ZIP}")
print(f"PROC_DIR: {PROC_DIR}")

for d in [PROC_DIR, TABLES_DIR, FIG_DIR]:
    os.makedirs(d, exist_ok=True)

print("Environment ready.")
print(f"  DATA_ZIP   : {DATA_ZIP}")
print(f"  PROC_DIR   : {PROC_DIR}")


Project root directory: /Users/boratonaj/Documents/GWU_Study_Course/Spring_Class_2026/responsible_AI/hmda-capstone-responsible
DATA_ZIP: /Users/boratonaj/Documents/GWU_Study_Course/Spring_Class_2026/responsible_AI/hmda-capstone-responsible/data/raw/2024_lar.zip
PROC_DIR: /Users/boratonaj/Documents/GWU_Study_Course/Spring_Class_2026/responsible_AI/hmda-capstone-responsible/data/processed
Environment ready.
  DATA_ZIP   : /Users/boratonaj/Documents/GWU_Study_Course/Spring_Class_2026/responsible_AI/hmda-capstone-responsible/data/raw/2024_lar.zip
  PROC_DIR   : /Users/boratonaj/Documents/GWU_Study_Course/Spring_Class_2026/responsible_AI/hmda-capstone-responsible/data/processed


## Section 1.1 — Source and Provenance

We document the data source and file hash before any processing begins.
This is audit evidence that we know exactly which version of the data we used.


In [2]:
# Compute SHA-256 hash of the zip file for provenance documentation
def file_sha256(path, chunk_size=8192):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

if os.path.exists(DATA_ZIP):
    sha = file_sha256(DATA_ZIP)
    print(f"SHA-256: {sha}")
    print(f"File size: {os.path.getsize(DATA_ZIP) / 1e9:.2f} GB")
    with open(os.path.join(os.getcwd(), "..", "data", "README.md"), "r") as f:
        content = f.read()
    content = content.replace("[INSERT SHA-256 HASH OF 2024_lar.zip OR 2024_lar.txt]", sha)
    with open(os.path.join(os.getcwd(), "..", "data", "README.md"), "w") as f:
        f.write(content)
    print("data/README.md updated with hash.")
else:
    print(f"WARNING: Data zip not found at {DATA_ZIP}")
    print("Please set DATA_ZIP to the correct path.")


SHA-256: 3f1578c027b1ae388e944492ac7b24d73a4deeb5da73ff731bebf34358a256a9
File size: 0.67 GB
data/README.md updated with hash.


## Section 1.2 — Raw Load and Row Counts

We load the full 2024 HMDA LAR in chunks to handle the ~4.6GB file size efficiently.
We track row counts at every filtering step.
This table is the first piece of audit evidence in the data pipeline.

**Key columns we rely on:**
- `action_taken`: the label we will predict
- `derived_race`, `derived_sex`, `derived_ethnicity`: HMDA-derived protected attributes
- `income`, `loan_amount`, `property_value`, `debt_to_income_ratio`: core creditworthiness features
- `census_tract`, `tract_minority_population_percent`: geographic features with proxy-risk concern


In [3]:
# Load 2024 HMDA LAR from zip — pipe-delimited
# We use chunksize to handle the large file efficiently
# For development, use SAMPLE_FRAC < 1.0 to iterate quickly

SAMPLE_FRAC = 1.0       # Set to e.g. 0.2 for fast development runs
CHUNKSIZE = 200_000
GEO_HOLDOUT_STATES = ["CA"]  # Hold out California as geographic drift test set
                              # Decision R-003: selected for size and demographic diversity

print(f"Loading 2024 HMDA LAR from {DATA_ZIP}")
print(f"Sample fraction: {SAMPLE_FRAC}")
print(f"Geographic holdout state(s): {GEO_HOLDOUT_STATES}")
print("This may take several minutes for the full dataset...")

chunks = []
with zipfile.ZipFile(DATA_ZIP) as z:
    with z.open("2024_lar.txt") as f:
        for chunk in pd.read_csv(
            f,
            sep="|",
            chunksize=CHUNKSIZE,
            dtype=str,
            low_memory=False,
        ):
            chunks.append(chunk)

raw_df = pd.concat(chunks, ignore_index=True)

if SAMPLE_FRAC < 1.0:
    raw_df = raw_df.sample(frac=SAMPLE_FRAC, random_state=SEED).reset_index(drop=True)
    print(f"  [DEVELOPMENT SAMPLE] Using {len(raw_df):,} rows ({SAMPLE_FRAC:.0%})")

n_raw = len(raw_df)
print(f"\nRaw rows loaded: {n_raw:,}")
print(f"Columns: {len(raw_df.columns)}")
print(f"action_taken value counts:")
print(raw_df["action_taken"].value_counts().sort_index())


Loading 2024 HMDA LAR from /Users/boratonaj/Documents/GWU_Study_Course/Spring_Class_2026/responsible_AI/hmda-capstone-responsible/data/raw/2024_lar.zip
Sample fraction: 1.0
Geographic holdout state(s): ['CA']
This may take several minutes for the full dataset...

Raw rows loaded: 12,259,151
Columns: 99
action_taken value counts:
action_taken
1    6197088
2     361225
3    2103460
4    1538268
5     581937
6    1276504
7      47313
8     153356
Name: count, dtype: int64


## Section 1.3 — Label Construction

Apply the capstone label rule. This uses `src/labels.py` — the single source of truth.

**Rule:** `{1,2}  1`, `{3}  0`, all other values filtered out.
**Rationale:** Documented in `docs/decision_log.md` (D-001) and `docs/00_system_card.md`.


In [4]:
# Show label distribution before filtering
dist_table = label_distribution_table(raw_df)
print("Label distribution table (before filtering):")
display(dist_table)
dist_table.to_csv(os.path.join(TABLES_DIR, "label_distribution.csv"), index=False)

Label distribution table (before filtering):


,action_taken,count,meaning,label_assigned,retained,pct_of_total
0,1,6197088,Loan originated → y=1,1.0,True,0.5055
1,2,361225,"Application approved, not accepted → y=1",1.0,True,0.0295
2,3,2103460,Application denied → y=0,0.0,True,0.1716
3,4,1538268,Application withdrawn by applicant → DROPPED,NaN,False,0.1255
4,5,581937,File closed for incompleteness → DROPPED,NaN,False,0.0475
5,6,1276504,Purchased loan → DROPPED,NaN,False,0.1041
6,7,47313,Preapproval request denied → DROPPED,NaN,False,0.0039
7,8,153356,Preapproval request approved but not accepted ...,NaN,False,0.0125


In [5]:
# Apply label rule
from src.labels import apply_label

df = apply_label(raw_df)
print(f"\nClass balance check:")
print(f"  y=1 (originated/approved): {(df['y']==1).sum():,}  ({df['y'].mean():.3f})")
print(f"  y=0 (denied):              {(df['y']==0).sum():,}  ({1-df['y'].mean():.3f})")

LABEL CONSTRUCTION LOG
  Raw rows:             12,259,151
  Mappable rows:         8,661,773  (70.7%)
  Dropped rows:          3,597,378  (action_taken not in {1,2,3})
  Final rows:            8,661,773
  Positive (y=1):        6,558,313  (0.757)
  Negative (y=0):        2,103,460  (0.243)

Class balance check:
  y=1 (originated/approved): 6,558,313  (0.757)
  y=0 (denied):              2,103,460  (0.243)


## Section 1.4 — Leakage Quarantine

Remove all post-decision features. These are ONLY populated after the lending decision
and would cause target leakage. This is enforced in `src/leakage.py`.

**Features removed:** `interest_rate`, `rate_spread`, `denial_reason_*`, `total_loan_costs`,
`origination_charges`, `discount_points`, `lender_credits`, `prepayment_penalty_term`,
`intro_rate_period`, `lei`, `activity_year`


In [6]:
# Remove post-decision features
df = remove_leakage(df)

# Also show proxy-risk table
proxy_risk_df = get_proxy_risk_table()
print("\nProxy-risk flagged features (retained but monitored):")
display(proxy_risk_df)
proxy_risk_df.to_csv(os.path.join(TABLES_DIR, "proxy_risk_features.csv"), index=False)


LEAKAGE REMOVAL LOG
  Features removed (15):
    - interest_rate
    - rate_spread
    - total_loan_costs
    - total_points_and_fees
    - origination_charges
    - discount_points
    - lender_credits
    - prepayment_penalty_term
    - intro_rate_period
    - denial_reason_1
    - denial_reason_2
    - denial_reason_3
    - denial_reason_4
    - lei
    - activity_year
  Not in dataset (OK, 1): ['loan_costs']
  Columns before: 100
  Columns after:  85

Proxy-risk flagged features (retained but monitored):


,feature,risk_level,justification
0,census_tract,High,Census tract encodes residential segregation p...
1,tract_minority_population_percent,High,Direct demographic encoding of tract-level min...
2,derived_msa_md,Medium,MSA/MD encodes regional income and demographic...
3,ffiec_msa_md_median_family_income,Medium,Tract-level income reflects structural income ...
4,tract_to_msa_income_percentage,Medium,Relative income of tract vs. MSA; encodes neig...
5,income,Medium,Applicant income is a legitimate creditworthin...
6,applicant_age,Low,Age is a protected attribute under ECOA. Monit...
7,state_code,Medium,State encodes regional demographic composition...
8,county_code,Medium,County-level demographic encoding similar to MSA.


## Section 1.5 — Protected Attribute Handling

`derived_race`, `derived_sex`, and `derived_ethnicity` are retained as protected attribute
columns. They are **NOT** used as model input features but are used in fairness analysis.

We also create a `race_sex_intersection` column for intersectional analysis.

**Decision D-008:** Applicants who selected "Not Provided" or "Not Applicable" are
retained as a distinct group rather than being dropped.


In [7]:
# Check protected attribute coverage
print("Protected attribute value counts:")
for col in ["derived_race", "derived_sex", "derived_ethnicity", "applicant_age"]:
    if col in df.columns:
        print(f"\n  {col}:")
        print(df[col].value_counts())

# Create intersectional column
df["race_sex_intersection"] = df["derived_race"].astype(str) + " × " + df["derived_sex"].astype(str)

# Count intersection cells
intersection_counts = df["race_sex_intersection"].value_counts()
print(f"\nIntersectional cells (race × sex): {len(intersection_counts)}")
print(f"Cells with n >= 30: {(intersection_counts >= 30).sum()}")
print(f"Cells with n < 30 (will be suppressed): {(intersection_counts < 30).sum()}")


Protected attribute value counts:

  derived_race:
derived_race
White                                        5550899
Race Not Available                           1526051
Black or African American                     762461
Asian                                         524983
Joint                                         189064
American Indian or Alaska Native               63158
2 or more minority races                       21855
Native Hawaiian or Other Pacific Islander      21117
Free Form Text Only                             2185
Name: count, dtype: int64

  derived_sex:
derived_sex
Joint                3003110
Male                 2934484
Female               1951979
Sex Not Available     772200
Name: count, dtype: int64

  derived_ethnicity:
derived_ethnicity
Not Hispanic or Latino     5968141
Ethnicity Not Available    1410834
Hispanic or Latino         1055891
Joint                       223382
Free Form Text Only           3525
Name: count, dtype: int64

  applicant_age:
appl

## Section 1.6 — Feature Engineering

Apply transformations from `src/features.py`:
- DTI string buckets -> numeric midpoints
- Outlier capping on loan_amount, income, property_value
- One-hot encoding of categorical features
- Median imputation for remaining NaN

Protected attributes are stored separately and NOT included in the feature matrix.
Applicant sex and applicant age are retained for fairness analysis only.


In [8]:
# Race percentages
# Compute proportions
race_dist = df["derived_race"].value_counts(normalize=True) * 100
# Print in desired format
for race, pct in race_dist.items():
    print(f"{race} Race: {pct:.2f}%")

White Race: 64.09%
Race Not Available Race: 17.62%
Black or African American Race: 8.80%
Asian Race: 6.06%
Joint Race: 2.18%
American Indian or Alaska Native Race: 0.73%
2 or more minority races Race: 0.25%
Native Hawaiian or Other Pacific Islander Race: 0.24%
Free Form Text Only Race: 0.03%


### Data Processing and Features Selections

In [9]:
# Feature groups for the HMDA subset
numerical_variables = [
    "income",
    "debt_to_income_ratio",
    "combined_loan_to_value_ratio",
    "loan_amount",
    "property_value",
    "interest_rate",
    "rate_spread",
    "loan_term",
    "total_loan_costs",
    "origination_charges",
    "discount_points",
    "lender_credits",
    "total_units",
]

datetime_variables = [
    # No datetime columns are selected in prepare_hmda_subset; keep empty unless you add date fields
]
# Categorical variables (including protected attributes and intersectional column)
categorical_variables = [
    "loan_type",
    "loan_purpose",
    "lien_status",
    "occupancy_type",
    "construction_method",
    "negative_amortization",
    "interest_only_payment",
    "balloon_payment",
    "other_nonamortizing_features",
    "prepayment_penalty_term",
    "state_code",
    "derived_race",
    "derived_sex",
    "derived_ethnicity",
    "applicant_sex",
    "race_sex_intersection",
    "applicant_age"
]

# ── Type conversions ───────────────────────────────────────────
# Numerical conversion
for col in numerical_variables:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Datetime conversion (if any)
for col in datetime_variables:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

# Convert categorical variables
for col in categorical_variables:
    if col in df.columns:
        df[col] = df[col].astype("category")

# Build feature dataframe from available columns (preserve order)
# Target variable
target = "y"
feature_cols = [c for c in (numerical_variables + datetime_variables + categorical_variables) if c in df.columns]
df_features = df[feature_cols + [target]].copy()

# Target variable
target = "y"

# Python equivalent of R's glimpse(df) and summary(df)
print(df_features.info())
print(df_features.describe(include='all').T)
print(f"Target variable distribution:")
print(f"  y=1 (originated/approved): {(df_features[target]==1).sum():,}  ({df_features[target].mean():.3f})")
print(f"  y=0 (denied):              {(df_features[target]==0).sum():,}  ({1-df_features[target].mean():.3f})")
print(f"  [NOTE] Model target is 'y' (binary: 1=approved, 0=denied)")

<class 'pandas.DataFrame'>
RangeIndex: 8661773 entries, 0 to 8661772
Data columns (total 24 columns):
 #   Column                        Dtype   
---  ------                        -----   
 0   income                        float64 
 1   debt_to_income_ratio          float64 
 2   combined_loan_to_value_ratio  float64 
 3   loan_amount                   int64   
 4   property_value                float64 
 5   loan_term                     float64 
 6   total_units                   float64 
 7   loan_type                     category
 8   loan_purpose                  category
 9   lien_status                   category
 10  occupancy_type                category
 11  construction_method           category
 12  negative_amortization         category
 13  interest_only_payment         category
 14  balloon_payment               category
 15  other_nonamortizing_features  category
 16  state_code                    category
 17  derived_race                  category
 18  derived_sex  

In [10]:
# show first few rows of the processed dataframe
df_features.head(5)

,income,debt_to_income_ratio,combined_loan_to_value_ratio,loan_amount,property_value,loan_term,total_units,loan_type,loan_purpose,lien_status,...,balloon_payment,other_nonamortizing_features,state_code,derived_race,derived_sex,derived_ethnicity,applicant_sex,race_sex_intersection,applicant_age,y
0,31.0,NaN,NaN,15000,NaN,NaN,1.0,1,31,1,...,1111,1111,GA,Black or African American,Female,Not Hispanic or Latino,2,Black or African American × Female,55-64,1
1,43.0,NaN,NaN,15000,NaN,NaN,1.0,1,1,1,...,1111,1111,GA,White,Male,Hispanic or Latino,1,White × Male,25-34,1
2,69.0,NaN,NaN,15000,NaN,NaN,1.0,1,31,1,...,1111,1111,GA,White,Male,Not Hispanic or Latino,1,White × Male,35-44,1
3,NaN,NaN,NaN,405000,NaN,NaN,1.0,1,31,1,...,1111,1111,FL,Race Not Available,Sex Not Available,Ethnicity Not Available,4,Race Not Available × Sex Not Available,8888,1
4,256.0,NaN,NaN,405000,NaN,NaN,NaN,1,31,1,...,1111,1111,GA,White,Male,Not Hispanic or Latino,1,White × Male,35-44,1


In [11]:
df_features.info()

<class 'pandas.DataFrame'>
RangeIndex: 8661773 entries, 0 to 8661772
Data columns (total 24 columns):
 #   Column                        Dtype   
---  ------                        -----   
 0   income                        float64 
 1   debt_to_income_ratio          float64 
 2   combined_loan_to_value_ratio  float64 
 3   loan_amount                   int64   
 4   property_value                float64 
 5   loan_term                     float64 
 6   total_units                   float64 
 7   loan_type                     category
 8   loan_purpose                  category
 9   lien_status                   category
 10  occupancy_type                category
 11  construction_method           category
 12  negative_amortization         category
 13  interest_only_payment         category
 14  balloon_payment               category
 15  other_nonamortizing_features  category
 16  state_code                    category
 17  derived_race                  category
 18  derived_sex  

In [12]:
df_features['applicant_age'].value_counts().sort_index()

applicant_age
25-34    1692175
35-44    2057697
45-54    1794403
55-64    1410085
65-74     826709
8888      227417
<25       307049
>74       346238
Name: count, dtype: int64

In [22]:
from src.features import build_feature_matrix, PROTECTED_ATTRS, NUM_FEATURES, CAT_FEATURES

# Store protected attributes and target separately before feature engineering
PROTECTED_COLS = [
    "derived_race",
    "derived_sex",
    "derived_ethnicity",
    "applicant_sex",
    "applicant_age",
    "race_sex_intersection",
]
# Note: "action_taken" = y
META_COLS = ["y", "state_code"] + PROTECTED_COLS

# Build feature matrix (no protected attrs, no target)
print("Building feature matrix...")
X = build_feature_matrix(df_features)
y = df_features["y"].values
meta = df_features[META_COLS].reset_index(drop=True)

print(f"Feature matrix shape: {X.shape}")
print(f"Features included: {len(X.columns)}")
print(f"Missing values in X: {X.isnull().sum().sum()}")


Building feature matrix...
Feature matrix shape: (8661773, 30)
Features included: 30
Missing values in X: 8661773


## Section 1.7 — Train/Val/Test Splits + Geographic Holdout

**Split strategy (Decision D-003):**
- Geographic holdout: California (`state_code == 'CA'`) held out entirely for drift testing
- Remaining data: stratified random 70/15/15 (train/val/test) by `y`

Stratification ensures class balance is preserved in all splits.
Geographic holdout simulates prospective distribution shift.


In [ ]:
# Geographic holdout: hold out California
geo_mask = df_features["state_code"].isin(GEO_HOLDOUT_STATES)
non_geo_mask = ~geo_mask

X_geo    = X[geo_mask].reset_index(drop=True)
y_geo    = y[geo_mask]
meta_geo = meta[geo_mask].reset_index(drop=True)

X_main    = X[non_geo_mask].reset_index(drop=True)
y_main    = y[non_geo_mask]
meta_main = meta[non_geo_mask].reset_index(drop=True)

print(f"Geographic holdout ({GEO_HOLDOUT_STATES}): {len(X_geo):,} rows")
print(f"Remaining (non-holdout): {len(X_main):,} rows")



Geographic holdout (['CA']): 727,912 rows
Remaining (non-holdout): 7,933,861 rows

Split summary:
  Train:        5,553,702  (positive rate: 0.758)
  Validation:   1,190,079  (positive rate: 0.758)
  Test:         1,190,080  (positive rate: 0.758)
  Geo holdout:    727,912  (positive rate: 0.752)

Stratification check:
  train positive rate: 0.7576
  val positive rate: 0.7576
  test positive rate: 0.7576


## Section 1.8 — Export Processed Files

Export all splits to parquet format. These files are the inputs for all subsequent notebooks.


In [ ]:
# Stratified train/val/test split on non-holdout data
# 70% train, 15% val, 15% test — roughly 70/15/15
X_train_full, X_test, y_train_full, y_test, meta_train_full, meta_test = (
    train_test_split(X_main, y_main, meta_main, test_size=0.15, random_state=SEED, stratify=y_main)
)
X_train, X_val, y_train, y_val, meta_train, meta_val = (
    train_test_split(X_train_full, y_train_full, meta_train_full,
                     test_size=0.15/0.85, random_state=SEED, stratify=y_train_full)
)

print(f"\nSplit summary:")
print(f"  Train:       {len(X_train):>10,}  (positive rate: {y_train.mean():.3f})")
print(f"  Validation:  {len(X_val):>10,}  (positive rate: {y_val.mean():.3f})")
print(f"  Test:        {len(X_test):>10,}  (positive rate: {y_test.mean():.3f})")
print(f"  Geo holdout: {len(X_geo):>10,}  (positive rate: {y_geo.mean():.3f})")

# Confirm stratification
print(f"\nStratification check:")
for name, ys in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(f"  {name} positive rate: {ys.mean():.4f}")

# Export processed splits to Parquet files, including features, target, and metadata columns
def export_split(X, y, meta, prefix, path):
    out = X.copy()
    out["y"] = y
    for col in meta.columns:
        out[col] = meta[col].values
    fpath = os.path.join(path, f"{prefix}.parquet")
    out.to_parquet(fpath, index=False)
    print(f"  Saved: {fpath}  ({len(out):,} rows x {len(out.columns)} cols)")
    return out

print("Exporting processed files...")
train_df = export_split(X_train, y_train, meta_train, "train", PROC_DIR)
val_df   = export_split(X_val,   y_val,   meta_val,   "val",   PROC_DIR)
test_df  = export_split(X_test,  y_test,  meta_test,  "test",  PROC_DIR)
geo_df   = export_split(X_geo,   y_geo,   meta_geo,   "geo_holdout", PROC_DIR)

# Export feature column list for use in downstream notebooks
feature_cols = X_train.columns.tolist()
with open(os.path.join(PROC_DIR, "feature_columns.txt"), "w") as f:
    f.write("\n".join(feature_cols))
print(f"\nFeature column list saved: {len(feature_cols)} features")

# Data dictionary
data_dict = pd.DataFrame({
    "feature": feature_cols,
    "dtype": [str(X_train[c].dtype) for c in feature_cols],
    "n_missing": [X_train[c].isnull().sum() for c in feature_cols],
    "mean": [X_train[c].mean() if X_train[c].dtype in ["float64", "int64"] else None for c in feature_cols],
})
data_dict.to_csv(os.path.join(TABLES_DIR, "data_dictionary.csv"), index=False)
print(f"Data dictionary saved.")


Exporting processed files...
  Saved: /Users/boratonaj/Documents/GWU_Study_Course/Spring_Class_2026/responsible_AI/hmda-capstone-responsible/data/processed/train.parquet  (5,553,702 rows x 38 cols)
  Saved: /Users/boratonaj/Documents/GWU_Study_Course/Spring_Class_2026/responsible_AI/hmda-capstone-responsible/data/processed/val.parquet  (1,190,079 rows x 38 cols)
  Saved: /Users/boratonaj/Documents/GWU_Study_Course/Spring_Class_2026/responsible_AI/hmda-capstone-responsible/data/processed/test.parquet  (1,190,080 rows x 38 cols)
  Saved: /Users/boratonaj/Documents/GWU_Study_Course/Spring_Class_2026/responsible_AI/hmda-capstone-responsible/data/processed/geo_holdout.parquet  (727,912 rows x 38 cols)

Feature column list saved: 30 features
Data dictionary saved.


## Section 1.9 — Row Count Audit Table

Final documentation of the data filtering pipeline.
This table is audit evidence that every filtering decision is accounted for.


In [25]:
audit_rows = {
    "Step": [
        "1. Raw HMDA LAR rows",
        "2. action_taken in {1,2,3} (after label filter)",
        "3. Leakage features removed (same rows)",
        "4. Feature matrix built",
        "5. Train split (70%)",
        "6. Validation split (15%)",
        "7. Test split (15%)",
        "8. Geographic holdout (CA)",
    ],
    "Row Count": [
        n_raw,
        len(df),
        len(df),
        len(X_main) + len(X_geo),
        len(X_train),
        len(X_val),
        len(X_test),
        len(X_geo),
    ],
    "Positive Rate": [
        None,
        round(df["y"].mean(), 3),
        round(df["y"].mean(), 3),
        None,
        round(y_train.mean(), 3),
        round(y_val.mean(), 3),
        round(y_test.mean(), 3),
        round(y_geo.mean(), 3),
    ],
    "Notes": [
        "Raw HMDA 2024 LAR",
        f"Dropped {n_raw - len(df):,} rows with action_taken in {{4,5,6,7,8}}",
        f"Removed {len(df.columns) - len(X_train.columns)} post-decision features",
        "Feature engineering applied",
        "Stratified by y",
        "Stratified by y",
        "Stratified by y",
        "State = CA — geographic drift test set",
    ]
}
audit_df = pd.DataFrame(audit_rows)
display(audit_df)
audit_df.to_csv(os.path.join(TABLES_DIR, "data_pipeline_audit.csv"), index=False)
print("\nData pipeline audit table saved.")
print("\n Notebook 01 complete. All processed files saved to data/processed/")


,Step,Row Count,Positive Rate,Notes
0,1. Raw HMDA LAR rows,12259151,NaN,Raw HMDA 2024 LAR
1,"2. action_taken in {1,2,3} (after label filter)",8661773,0.757,"Dropped 3,597,378 rows with action_taken in {4..."
2,3. Leakage features removed (same rows),8661773,0.757,Removed 56 post-decision features
3,4. Feature matrix built,8661773,NaN,Feature engineering applied
4,5. Train split (70%),5553702,0.758,Stratified by y
5,6. Validation split (15%),1190079,0.758,Stratified by y
6,7. Test split (15%),1190080,0.758,Stratified by y
7,8. Geographic holdout (CA),727912,0.752,State = CA — geographic drift test set



Data pipeline audit table saved.

 Notebook 01 complete. All processed files saved to data/processed/
